# BERT multi-label classification

Task multi-label: quindi sigmoid + `BCEWithLogitsLoss` invece di softmax/argmax.

In [12]:

# %pip install transformers


In [1]:
import os
# Deve essere impostato prima di importare torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
HF_HOME = os.path.abspath(".hf_home")
HF_HUB_CACHE = os.path.abspath(".hf_cache")
HF_XET_CACHE = os.path.abspath(".hf_xet")
os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE
os.environ["HF_XET_CACHE"] = HF_XET_CACHE
os.environ["HF_HUB_DISABLE_XET"] = "1"
for cache_path in [HF_HOME, HF_HUB_CACHE, HF_XET_CACHE]:
    os.makedirs(cache_path, exist_ok=True)

import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, hamming_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from tqdm.auto import tqdm


/home/alicemusso/miniconda3/envs/iml_second_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


Device: cuda
NVIDIA A40


## Caricamento dati (solo abstract)


In [3]:
DATA_PATH = "CleanDataset.csv"
usecols = ["abstract_clean", "subjareas"]

df = pd.read_csv(DATA_PATH, usecols=usecols)
df = df.dropna(subset=["abstract_clean", "subjareas"]).copy()
print(df.shape)
df.head(2)


(38711, 2)


,subjareas,abstract_clean
0,MULT,data presented originally collected frontiers ...
1,MULT,easy method combined gel casting physical foam...


In [4]:
def safe_text(value):
    return "" if pd.isna(value) else str(value)

df["text"] = df["abstract_clean"].apply(safe_text)
df = df[df["text"].str.split().str.len() > 5].copy()
print(df.shape)
df[["text", "subjareas"]].head(2)


(38711, 3)


,text,subjareas
0,data presented originally collected frontiers ...,MULT
1,easy method combined gel casting physical foam...,MULT


## Label 18-classi

Come LSTM

In [5]:
def parse_labels(value):
    return [label.strip() for label in str(value).split(",") if label.strip()]

label_mapping_18 = {
    "VETE": "MEDI", "HEAL": "MEDI", "DENT": "MEDI", "NURS": "MEDI",
    "ECON": "SOCI", "ARTS": "SOCI", "BUSI": "SOCI", "DECI": "SOCI",
    "MATH": "MULT",
}

def remap_labels_18(label_list):
    return sorted(set(label_mapping_18.get(label, label) for label in label_list))

df["labels_list"] = df["subjareas"].apply(parse_labels)
df["labels_18_list"] = df["labels_list"].apply(remap_labels_18)

mlb_18 = MultiLabelBinarizer()
y_18 = mlb_18.fit_transform(df["labels_18_list"]).astype(np.float32)
labels_18 = mlb_18.classes_

print(f"Numero classi: {len(labels_18)}")
print(labels_18)
print(y_18.shape)


Numero classi: 18
['AGRI' 'BIOC' 'CENG' 'CHEM' 'COMP' 'EART' 'ENER' 'ENGI' 'ENVI' 'IMMU'
 'MATE' 'MEDI' 'MULT' 'NEUR' 'PHAR' 'PHYS' 'PSYC' 'SOCI']
(38711, 18)


In [6]:
label_counts_18 = pd.Series(y_18.sum(axis=0), index=labels_18).sort_values(ascending=False)
label_counts_18


MEDI    9885.0
BIOC    8196.0
ENVI    6166.0
ENGI    5849.0
SOCI    5180.0
AGRI    3942.0
MATE    3918.0
PHYS    3604.0
NEUR    3255.0
IMMU    3168.0
COMP    2919.0
ENER    2796.0
EART    2743.0
CHEM    2703.0
PHAR    2205.0
CENG    2139.0
PSYC    1774.0
MULT    1540.0
dtype: float32

## Split train / validation / test


In [7]:
X = df["text"].astype(str).tolist()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y_18,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
)

print(f"train: {len(X_train)}")
print(f"val:   {len(X_val)}")
print(f"test:  {len(X_test)}")


train: 24387
val:   2710
test:  11614


## Class weights / pos_weight

Le classi meno frequenti ricevono un peso maggiore nella `BCEWithLogitsLoss`, cosi gli errori sui positivi rari contano di piu durante il training.


In [8]:
def compute_pos_weight(train_labels):
    train_labels = np.asarray(train_labels, dtype=np.float32)
    positive_counts = train_labels.sum(axis=0)
    negative_counts = train_labels.shape[0] - positive_counts
    pos_weight = np.divide(
        negative_counts,
        positive_counts,
        out=np.zeros_like(positive_counts, dtype=np.float32),
        where=positive_counts > 0,
    ).astype(np.float32)

    weights_df = pd.DataFrame({
        "label": labels_18,
        "positive_count": positive_counts.astype(int),
        "negative_count": negative_counts.astype(int),
        "pos_weight": np.round(pos_weight, 3),
    }).sort_values("pos_weight", ascending=False)
    return pos_weight, weights_df

pos_weight_18, class_weights_df = compute_pos_weight(y_train)
class_weights_df


,label,positive_count,negative_count,pos_weight
12,MULT,994,23393,23.534000
16,PSYC,1093,23294,21.312000
14,PHAR,1368,23019,16.827000
2,CENG,1386,23001,16.594999
5,EART,1719,22668,13.187000
3,CHEM,1730,22657,13.097000
6,ENER,1781,22606,12.693000
4,COMP,1807,22580,12.496000
9,IMMU,2023,22364,11.055000
13,NEUR,2028,22359,11.025000


## Tokenizer e Dataset



In [9]:

print(f"Device corrente: {device}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


Device corrente: cuda
CUDA disponibile: True
NVIDIA A40


In [12]:
BERT_BASE_MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 512
BATCH_SIZE = 8
CACHE_DIR = HF_HUB_CACHE

tokenizer = AutoTokenizer.from_pretrained(BERT_BASE_MODEL_NAME, cache_dir=CACHE_DIR)


In [13]:
class ArticleBertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def make_dataloaders(tokenizer, max_length=MAX_LENGTH, batch_size=BATCH_SIZE):
    train_dataset = ArticleBertDataset(X_train, y_train, tokenizer, max_length)
    val_dataset = ArticleBertDataset(X_val, y_val, tokenizer, max_length)
    test_dataset = ArticleBertDataset(X_test, y_test, tokenizer, max_length)

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_dataloader, val_dataloader, test_dataloader

train_dataloader, val_dataloader, test_dataloader = make_dataloaders(tokenizer)


In [14]:
EPOCHS = 10

def make_model_components(
    model_name,
    train_labels,
    train_dataloader,
    epochs=EPOCHS,
    lr=2e-5,
    pos_weight_values=None,
):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        cache_dir=CACHE_DIR,
        num_labels=len(labels_18),
        problem_type="multi_label_classification",
        output_attentions=False,
        output_hidden_states=False,
    )
    model.to(device)

    if pos_weight_values is None:
        pos_weight_values, _ = compute_pos_weight(train_labels)
    pos_weight = torch.as_tensor(pos_weight_values, dtype=torch.float32, device=device)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-8)

    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps,
    )
    return model, criterion, optimizer, scheduler

bert_base_model, bert_base_criterion, bert_base_optimizer, bert_base_scheduler = make_model_components(
    BERT_BASE_MODEL_NAME,
    y_train,
    train_dataloader,
    epochs=EPOCHS,
    pos_weight_values=pos_weight_18,
)

print(bert_base_model.config)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2887.14it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_11": 11,
    "LABEL_12": 12,
    "LABEL_13": 13,
    "LABEL_14": 14,
    "LABEL_15": 15,
    "LAB

In [15]:
thresholds = np.arange(0.1, 0.9, 0.05)

def batch_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}

def multilabel_metrics(y_true, y_pred):
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }

def threshold_search(y_true, y_prob, thresholds=thresholds):
    rows = []
    for threshold in thresholds:
        threshold = float(np.round(threshold, 2))
        y_pred = (y_prob >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            **multilabel_metrics(y_true, y_pred),
        })
    return pd.DataFrame(rows)

def select_best_threshold(y_true, y_prob, thresholds=thresholds, metric="micro_f1"):
    scores = threshold_search(y_true, y_prob, thresholds=thresholds)
    best_idx = scores[metric].idxmax()
    return float(scores.loc[best_idx, "threshold"]), scores

@torch.no_grad()
def evaluate(model, dataloader, criterion, threshold=None, thresholds=thresholds, selection_metric="micro_f1"):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_targets = []

    for batch in dataloader:
        batch = batch_to_device(batch, device)
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )
        loss = criterion(outputs.logits, batch["labels"])
        total_loss += loss.item()

        probs = torch.sigmoid(outputs.logits).detach().cpu().numpy()
        targets = batch["labels"].detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets)

    y_prob = np.vstack(all_probs)
    y_true = np.vstack(all_targets)
    threshold_scores = None

    if threshold is None:
        threshold, threshold_scores = select_best_threshold(
            y_true,
            y_prob,
            thresholds=thresholds,
            metric=selection_metric,
        )

    y_pred = (y_prob >= threshold).astype(int)
    metrics = multilabel_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(dataloader)
    metrics["threshold"] = float(threshold)
    return metrics, y_true, y_pred, y_prob, threshold_scores


In [16]:
def train_model(
    model,
    train_dataloader,
    val_dataloader,
    criterion,
    optimizer,
    scheduler,
    epochs=EPOCHS,
    thresholds=thresholds,
    selection_metric="micro_f1",
    experiment_name="model",
    patience=2,
    early_stopping_metric="val_micro_f1",
):
    history = []
    best_val_score = -np.inf
    best_state_dict = None
    best_epoch = 0
    early_stop_counter = 0

    for epoch in range(epochs):
        print(f"{experiment_name} | Epoch {epoch + 1} / {epochs} ========")
        model.train()
        total_train_loss = 0.0
        train_probs = []
        train_targets = []

        progress = tqdm(train_dataloader, desc=f"Training {experiment_name}", leave=False)
        for batch in progress:
            batch = batch_to_device(batch, device)

            optimizer.zero_grad()
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            loss = criterion(outputs.logits, batch["labels"])
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            probs = torch.sigmoid(outputs.logits).detach().cpu().numpy()
            targets = batch["labels"].detach().cpu().numpy()
            train_probs.append(probs)
            train_targets.append(targets)
            progress.set_postfix(loss=f"{loss.item():.4f}")

        y_train_prob = np.vstack(train_probs)
        y_train_true = np.vstack(train_targets)
        train_loss = total_train_loss / len(train_dataloader)

        val_metrics, _, _, _, _ = evaluate(
            model,
            val_dataloader,
            criterion,
            thresholds=thresholds,
            selection_metric=selection_metric,
        )
        selected_threshold = val_metrics["threshold"]

        y_train_pred = (y_train_prob >= selected_threshold).astype(int)
        train_metrics = multilabel_metrics(y_train_true, y_train_pred)

        row = {
            "experiment": experiment_name,
            "epoch": epoch + 1,
            "threshold": selected_threshold,
            "train_loss": train_loss,
            "train_micro_f1": train_metrics["micro_f1"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_micro_f1": val_metrics["micro_f1"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_hamming_loss": val_metrics["hamming_loss"],
        }
        history.append(row)

        print(
            f"threshold={row['threshold']:.2f} "
            f"train_loss={row['train_loss']:.4f} "
            f"train_micro_f1={row['train_micro_f1']:.4f} "
            f"val_loss={row['val_loss']:.4f} "
            f"val_micro_f1={row['val_micro_f1']:.4f} "
            f"val_macro_f1={row['val_macro_f1']:.4f}"
        )

        current_val_score = row[early_stopping_metric]
        if current_val_score > best_val_score:
            best_val_score = current_val_score
            best_epoch = epoch + 1
            early_stop_counter = 0
            best_state_dict = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            print(f"Best model updated at epoch {best_epoch} ({early_stopping_metric}={best_val_score:.4f}).")
        else:
            early_stop_counter += 1
            print(f"No improvement. Patience: {early_stop_counter}/{patience}")

        if early_stop_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print(f"Reloaded best model from epoch {best_epoch} ({early_stopping_metric}={best_val_score:.4f}).")

    return pd.DataFrame(history)


In [27]:
bert_base_training_stats = train_model(
    bert_base_model,
    train_dataloader,
    val_dataloader,
    bert_base_criterion,
    bert_base_optimizer,
    bert_base_scheduler,
    epochs=EPOCHS,
    experiment_name="BERT base",
)
bert_base_training_stats


BERT base | Epoch 1 / 10 ========


threshold=0.75 train_loss=0.6803 train_micro_f1=0.5885 val_loss=0.5710 val_micro_f1=0.6259 val_macro_f1=0.6110
Best model updated at epoch 1 (val_micro_f1=0.6259).
BERT base | Epoch 2 / 10 ========


threshold=0.80 train_loss=0.5064 train_micro_f1=0.6770 val_loss=0.5288 val_micro_f1=0.6637 val_macro_f1=0.6463
Best model updated at epoch 2 (val_micro_f1=0.6637).
BERT base | Epoch 3 / 10 ========


threshold=0.80 train_loss=0.4015 train_micro_f1=0.7333 val_loss=0.5523 val_micro_f1=0.6822 val_macro_f1=0.6604
Best model updated at epoch 3 (val_micro_f1=0.6822).
BERT base | Epoch 4 / 10 ========


threshold=0.80 train_loss=0.3141 train_micro_f1=0.7857 val_loss=0.5534 val_micro_f1=0.6827 val_macro_f1=0.6655
Best model updated at epoch 4 (val_micro_f1=0.6827).
BERT base | Epoch 5 / 10 ========


threshold=0.80 train_loss=0.2444 train_micro_f1=0.8285 val_loss=0.6802 val_micro_f1=0.6977 val_macro_f1=0.6752
Best model updated at epoch 5 (val_micro_f1=0.6977).
BERT base | Epoch 6 / 10 ========


threshold=0.70 train_loss=0.1895 train_micro_f1=0.8638 val_loss=0.7577 val_micro_f1=0.6936 val_macro_f1=0.6707
No improvement. Patience: 1/2
BERT base | Epoch 7 / 10 ========


threshold=0.65 train_loss=0.1462 train_micro_f1=0.8905 val_loss=0.8579 val_micro_f1=0.6951 val_macro_f1=0.6707
No improvement. Patience: 2/2
Early stopping triggered at epoch 7.
Reloaded best model from epoch 5 (val_micro_f1=0.6977).


,experiment,epoch,threshold,train_loss,train_micro_f1,train_macro_f1,val_loss,val_micro_f1,val_macro_f1,val_hamming_loss
0,BERT base,1,0.75,0.680341,0.588529,0.567869,0.570968,0.625899,0.611021,0.085281
1,BERT base,2,0.80,0.506383,0.677037,0.660829,0.528755,0.663660,0.646294,0.073391
2,BERT base,3,0.80,0.401490,0.733296,0.722178,0.552338,0.682162,0.660423,0.068122
3,BERT base,4,0.80,0.314143,0.785743,0.781765,0.553448,0.682687,0.665494,0.068758
4,BERT base,5,0.80,0.244435,0.828462,0.829686,0.680233,0.697702,0.675211,0.062567
5,BERT base,6,0.70,0.189533,0.863838,0.865869,0.757657,0.693622,0.670674,0.066175
6,BERT base,7,0.65,0.146187,0.890479,0.894169,0.857889,0.695128,0.670714,0.066318


In [28]:
bert_base_val_metrics, _, _, _, bert_base_threshold_scores = evaluate(
    bert_base_model,
    val_dataloader,
    bert_base_criterion,
    thresholds=thresholds,
)
bert_base_best_threshold = bert_base_val_metrics["threshold"]

bert_base_test_metrics, y_test_true, bert_base_y_test_pred, bert_base_y_test_prob, _ = evaluate(
    bert_base_model,
    test_dataloader,
    bert_base_criterion,
    threshold=bert_base_best_threshold,
)

print(f"Best validation threshold: {bert_base_best_threshold:.2f}")
bert_base_test_metrics


Best validation threshold: 0.80


{'micro_f1': 0.6916381863750285,
 'macro_f1': 0.6672445034114084,
 'hamming_loss': 0.06473987333295066,
 'loss': 0.6945066872583933,
 'threshold': 0.8}

In [29]:
print(classification_report(
    y_test_true,
    bert_base_y_test_pred,
    target_names=labels_18,
    zero_division=0,
))


              precision    recall  f1-score   support

        AGRI       0.66      0.67      0.67      1219
        BIOC       0.74      0.66      0.70      2480
        CENG       0.44      0.57      0.49       599
        CHEM       0.51      0.61      0.56       791
        COMP       0.57      0.61      0.59       903
        EART       0.80      0.77      0.78       838
        ENER       0.66      0.69      0.67       794
        ENGI       0.67      0.66      0.67      1789
        ENVI       0.70      0.73      0.71      1866
        IMMU       0.68      0.77      0.72       946
        MATE       0.77      0.70      0.73      1187
        MEDI       0.81      0.75      0.78      3004
        MULT       0.46      0.57      0.51       440
        NEUR       0.74      0.84      0.79       999
        PHAR       0.59      0.61      0.60       675
        PHYS       0.64      0.68      0.66      1059
        PSYC       0.57      0.68      0.62       574
        SOCI       0.75    

## SciBERT

In [30]:

bert_base_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SCIBERT_MODEL_NAME = "allenai/scibert_scivocab_uncased"
scibert_tokenizer = AutoTokenizer.from_pretrained(SCIBERT_MODEL_NAME, cache_dir=CACHE_DIR)

scibert_train_dataloader, scibert_val_dataloader, scibert_test_dataloader = make_dataloaders(
    scibert_tokenizer,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
)

scibert_model, scibert_criterion, scibert_optimizer, scibert_scheduler = make_model_components(
    SCIBERT_MODEL_NAME,
    y_train,
    scibert_train_dataloader,
    epochs=EPOCHS,
    pos_weight_values=pos_weight_18,
)

print(scibert_model.config)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32559.64it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different ta

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9",
    "10": "LABEL_10",
    "11": "LABEL_11",
    "12": "LABEL_12",
    "13": "LABEL_13",
    "14": "LABEL_14",
    "15": "LABEL_15",
    "16": "LABEL_16",
    "17": "LABEL_17"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_10": 10,
    "LABEL_11": 11,
    "LABEL_12": 12,
    "LABEL_13": 13,
    "LABEL_14": 14,
    "LABEL_15": 15,
    "LABEL_16": 16,
    "LABEL_17": 17,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 

In [31]:
scibert_training_stats = train_model(
    scibert_model,
    scibert_train_dataloader,
    scibert_val_dataloader,
    scibert_criterion,
    scibert_optimizer,
    scibert_scheduler,
    epochs=EPOCHS,
    experiment_name="SciBERT",
)
scibert_training_stats


SciBERT | Epoch 1 / 10 ========


threshold=0.80 train_loss=0.6264 train_micro_f1=0.6088 val_loss=0.5029 val_micro_f1=0.6672 val_macro_f1=0.6514
Best model updated at epoch 1 (val_micro_f1=0.6672).
SciBERT | Epoch 2 / 10 ========


threshold=0.80 train_loss=0.4524 train_micro_f1=0.7102 val_loss=0.4781 val_micro_f1=0.6842 val_macro_f1=0.6688
Best model updated at epoch 2 (val_micro_f1=0.6842).
SciBERT | Epoch 3 / 10 ========


threshold=0.80 train_loss=0.3473 train_micro_f1=0.7701 val_loss=0.4983 val_micro_f1=0.7078 val_macro_f1=0.6907
Best model updated at epoch 3 (val_micro_f1=0.7078).
SciBERT | Epoch 4 / 10 ========


threshold=0.75 train_loss=0.2596 train_micro_f1=0.8249 val_loss=0.5650 val_micro_f1=0.7113 val_macro_f1=0.6879
Best model updated at epoch 4 (val_micro_f1=0.7113).
SciBERT | Epoch 5 / 10 ========


...

threshold=0.65 train_loss=0.0968 train_micro_f1=0.9306 val_loss=0.8447 val_micro_f1=0.7303 val_macro_f1=0.7077
Best model updated at epoch 7 (val_micro_f1=0.7303).
SciBERT | Epoch 8 / 10 ========


threshold=0.70 train_loss=0.0688 train_micro_f1=0.9549 val_loss=0.9275 val_micro_f1=0.7333 val_macro_f1=0.7108
Best model updated at epoch 8 (val_micro_f1=0.7333).
SciBERT | Epoch 9 / 10 ========


threshold=0.65 train_loss=0.0477 train_micro_f1=0.9705 val_loss=1.0127 val_micro_f1=0.7321 val_macro_f1=0.7086
No improvement. Patience: 1/2
SciBERT | Epoch 10 / 10 ========


threshold=0.70 train_loss=0.0365 train_micro_f1=0.9800 val_loss=1.0286 val_micro_f1=0.7327 val_macro_f1=0.7096
No improvement. Patience: 2/2
Early stopping triggered at epoch 10.
Reloaded best model from epoch 8 (val_micro_f1=0.7333).


,experiment,epoch,threshold,train_loss,train_micro_f1,train_macro_f1,val_loss,val_micro_f1,val_macro_f1,val_hamming_loss
0,SciBERT,1,0.80,0.626357,0.608792,0.591617,0.502893,0.667231,0.651419,0.072550
1,SciBERT,2,0.80,0.452441,0.710238,0.694656,0.478079,0.684181,0.668758,0.069250
2,SciBERT,3,0.80,0.347269,0.770084,0.760942,0.498285,0.707789,0.690661,0.062833
3,SciBERT,4,0.75,0.259592,0.824862,0.820568,0.565040,0.711282,0.687928,0.063059
4,SciBERT,5,0.65,0.188338,0.863674,0.862764,0.693361,0.722422,0.701500,0.060148
5,SciBERT,6,0.70,0.135446,0.905465,0.907429,0.783283,0.727395,0.703907,0.057975
6,SciBERT,7,0.65,0.096788,0.930583,0.933113,0.844684,0.730345,0.707746,0.057585
7,SciBERT,8,0.70,0.068803,0.954935,0.957514,0.927484,0.733281,0.710797,0.055597
8,SciBERT,9,0.65,0.047669,0.970472,0.971981,1.012683,0.732145,0.708613,0.055740
9,SciBERT,10,0.70,0.036521,0.980014,0.981530,1.028618,0.732687,0.709622,0.054838


In [32]:
scibert_val_metrics, _, _, _, scibert_threshold_scores = evaluate(
    scibert_model,
    scibert_val_dataloader,
    scibert_criterion,
    thresholds=thresholds,
)
scibert_best_threshold = scibert_val_metrics["threshold"]

scibert_test_metrics, scibert_y_test_true, scibert_y_test_pred, scibert_y_test_prob, _ = evaluate(
    scibert_model,
    scibert_test_dataloader,
    scibert_criterion,
    threshold=scibert_best_threshold,
)

print(f"Best validation threshold: {scibert_best_threshold:.2f}")
scibert_test_metrics


Best validation threshold: 0.70


{'micro_f1': 0.7187564481738772,
 'macro_f1': 0.6914059990246553,
 'hamming_loss': 0.05867918029963837,
 'loss': 1.0295082344327213,
 'threshold': 0.7}

In [33]:
print(classification_report(
    scibert_y_test_true,
    scibert_y_test_pred,
    target_names=labels_18,
    zero_division=0,
))


              precision    recall  f1-score   support

        AGRI       0.67      0.72      0.69      1219
        BIOC       0.73      0.76      0.75      2480
        CENG       0.55      0.54      0.55       599
        CHEM       0.57      0.58      0.58       791
        COMP       0.66      0.59      0.62       903
        EART       0.79      0.83      0.81       838
        ENER       0.67      0.69      0.68       794
        ENGI       0.69      0.69      0.69      1789
        ENVI       0.69      0.76      0.72      1866
        IMMU       0.76      0.70      0.73       946
        MATE       0.78      0.76      0.77      1187
        MEDI       0.82      0.79      0.80      3004
        MULT       0.56      0.53      0.55       440
        NEUR       0.77      0.85      0.81       999
        PHAR       0.60      0.67      0.63       675
        PHYS       0.66      0.70      0.68      1059
        PSYC       0.63      0.59      0.61       574
        SOCI       0.80    

## BERT vs SciBERT


In [25]:
comparison = pd.DataFrame([
    {"experiment": "BERT base", **bert_base_test_metrics},
    {"experiment": "SciBERT", **scibert_test_metrics},
])
comparison


NameError: name 'bert_base_test_metrics' is not defined

## Esperimento 3: SciBERT + TF-IDF su abstract e body

Questo esperimento resta separato dai precedenti. SciBERT riceve solo `abstract_clean`, mentre il TF-IDF viene calcolato su `abstract_clean` piu i primi token di `body_clean`. In questo modo il modello combina informazione contestuale densa e feature lessicali esplicite del documento.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModel
import torch.nn as nn

FUSION_MODEL_NAME = "allenai/scibert_scivocab_uncased"
FUSION_MAX_LENGTH = MAX_LENGTH
FUSION_BATCH_SIZE = BATCH_SIZE
TFIDF_BODY_TOKEN_LIMIT = 1000
TFIDF_MAX_FEATURES = 1000

fusion_usecols = ["abstract_clean", "body_clean", "subjareas"]
fusion_df = pd.read_csv(DATA_PATH, usecols=fusion_usecols)
fusion_df = fusion_df.dropna(subset=fusion_usecols).copy()

def first_n_tokens(value, n=TFIDF_BODY_TOKEN_LIMIT):
    text = safe_text(value)
    return " ".join(text.split()[:n])

fusion_df["bert_text"] = fusion_df["abstract_clean"].apply(safe_text)
fusion_df["tfidf_text"] = (
    fusion_df["body_clean"].apply(lambda value: first_n_tokens(value, TFIDF_BODY_TOKEN_LIMIT))
).str.strip()
fusion_df = fusion_df[fusion_df["bert_text"].str.split().str.len() > 5].copy()

fusion_df["labels_list"] = fusion_df["subjareas"].apply(parse_labels)
fusion_df["labels_18_list"] = fusion_df["labels_list"].apply(remap_labels_18)

fusion_mlb_18 = MultiLabelBinarizer()
fusion_y_18 = fusion_mlb_18.fit_transform(fusion_df["labels_18_list"]).astype(np.float32)
fusion_labels_18 = fusion_mlb_18.classes_

print(f"Dataset fusion: {fusion_df.shape}")
print(f"Classi fusion: {len(fusion_labels_18)}")
print(fusion_labels_18)
assert list(fusion_labels_18) == list(labels_18), "Ordine label diverso rispetto agli esperimenti precedenti"


Dataset fusion: (38711, 7)
Classi fusion: 18
['AGRI' 'BIOC' 'CENG' 'CHEM' 'COMP' 'EART' 'ENER' 'ENGI' 'ENVI' 'IMMU'
 'MATE' 'MEDI' 'MULT' 'NEUR' 'PHAR' 'PHYS' 'PSYC' 'SOCI']


In [18]:
fusion_indices = np.arange(len(fusion_df))

fusion_idx_train_full, fusion_idx_test, fusion_y_train_full, fusion_y_test = train_test_split(
    fusion_indices,
    fusion_y_18,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

fusion_idx_train, fusion_idx_val, fusion_y_train, fusion_y_val = train_test_split(
    fusion_idx_train_full,
    fusion_y_train_full,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
)

fusion_X_train_bert = fusion_df.iloc[fusion_idx_train]["bert_text"].tolist()
fusion_X_val_bert = fusion_df.iloc[fusion_idx_val]["bert_text"].tolist()
fusion_X_test_bert = fusion_df.iloc[fusion_idx_test]["bert_text"].tolist()

fusion_X_train_tfidf = fusion_df.iloc[fusion_idx_train]["tfidf_text"].tolist()
fusion_X_val_tfidf = fusion_df.iloc[fusion_idx_val]["tfidf_text"].tolist()
fusion_X_test_tfidf = fusion_df.iloc[fusion_idx_test]["tfidf_text"].tolist()

tfidf_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
)

tfidf_train = torch.from_numpy(
    tfidf_vectorizer.fit_transform(fusion_X_train_tfidf).astype(np.float32).toarray()
)
tfidf_val = torch.from_numpy(
    tfidf_vectorizer.transform(fusion_X_val_tfidf).astype(np.float32).toarray()
)
tfidf_test = torch.from_numpy(
    tfidf_vectorizer.transform(fusion_X_test_tfidf).astype(np.float32).toarray()
)

print(f"train: {len(fusion_X_train_bert)}")
print(f"val:   {len(fusion_X_val_bert)}")
print(f"test:  {len(fusion_X_test_bert)}")
print(f"TF-IDF dim: {tfidf_train.shape[1]}")


train: 24387
val:   2710
test:  11614
TF-IDF dim: 1000


In [19]:
class ArticleBertTfidfDataset(Dataset):
    def __init__(self, bert_texts, labels, tfidf_features, tokenizer, max_length):
        self.bert_texts = list(bert_texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tfidf_features = tfidf_features
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.bert_texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.bert_texts[idx],
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "tfidf_features": self.tfidf_features[idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }

def make_fusion_dataloaders(tokenizer, max_length=FUSION_MAX_LENGTH, batch_size=FUSION_BATCH_SIZE):
    train_dataset = ArticleBertTfidfDataset(
        fusion_X_train_bert, fusion_y_train, tfidf_train, tokenizer, max_length
    )
    val_dataset = ArticleBertTfidfDataset(
        fusion_X_val_bert, fusion_y_val, tfidf_val, tokenizer, max_length
    )
    test_dataset = ArticleBertTfidfDataset(
        fusion_X_test_bert, fusion_y_test, tfidf_test, tokenizer, max_length
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

fusion_tokenizer = AutoTokenizer.from_pretrained(FUSION_MODEL_NAME, cache_dir=CACHE_DIR)
fusion_train_dataloader, fusion_val_dataloader, fusion_test_dataloader = make_fusion_dataloaders(fusion_tokenizer)


In [20]:
class SciBertTfidfClassifier(nn.Module):
    def __init__(self, model_name, num_labels, tfidf_dim, tfidf_hidden_dim=128, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name, cache_dir=CACHE_DIR)
        hidden_size = self.bert.config.hidden_size

        self.bert_dropout = nn.Dropout(dropout)
        self.tfidf_proj = nn.Sequential(
            nn.Linear(tfidf_dim, tfidf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_size + tfidf_hidden_dim, num_labels)

    def forward(self, input_ids, attention_mask, tfidf_features):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        if getattr(outputs, "pooler_output", None) is not None:
            cls_output = outputs.pooler_output
        else:
            cls_output = outputs.last_hidden_state[:, 0]

        cls_output = self.bert_dropout(cls_output)
        tfidf_output = self.tfidf_proj(tfidf_features)
        combined = torch.cat([cls_output, tfidf_output], dim=1)
        return self.classifier(combined)

# Libera VRAM dagli esperimenti precedenti, se sono ancora caricati.
try:
    scibert_model.to("cpu")
except NameError:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()

FUSION_EPOCHS = 10

fusion_model = SciBertTfidfClassifier(
    FUSION_MODEL_NAME,
    num_labels=len(fusion_labels_18),
    tfidf_dim=tfidf_train.shape[1],
).to(device)

fusion_pos_weight_18, fusion_class_weights_df = compute_pos_weight(fusion_y_train)
fusion_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.as_tensor(fusion_pos_weight_18, dtype=torch.float32, device=device)
)
fusion_optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=2e-5, eps=1e-8)
fusion_total_steps = len(fusion_train_dataloader) * FUSION_EPOCHS
fusion_scheduler = get_linear_schedule_with_warmup(
    fusion_optimizer,
    num_warmup_steps=0,
    num_training_steps=fusion_total_steps,
)

fusion_model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21493.19it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SciBertTfidfClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

In [21]:
def fusion_batch_to_device(batch, device):
    return {key: value.to(device) for key, value in batch.items()}

@torch.no_grad()
def evaluate_fusion_model(model, dataloader, criterion, threshold=None, thresholds=thresholds, selection_metric="micro_f1"):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_targets = []

    for batch in dataloader:
        batch = fusion_batch_to_device(batch, device)
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            tfidf_features=batch["tfidf_features"],
        )
        loss = criterion(logits, batch["labels"])
        total_loss += loss.item()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        targets = batch["labels"].detach().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets)

    y_prob = np.vstack(all_probs)
    y_true = np.vstack(all_targets)
    threshold_scores = None

    if threshold is None:
        threshold, threshold_scores = select_best_threshold(
            y_true,
            y_prob,
            thresholds=thresholds,
            metric=selection_metric,
        )

    y_pred = (y_prob >= threshold).astype(int)
    metrics = multilabel_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(dataloader)
    metrics["threshold"] = float(threshold)
    return metrics, y_true, y_pred, y_prob, threshold_scores

def train_fusion_model(
    model,
    train_dataloader,
    val_dataloader,
    criterion,
    optimizer,
    scheduler,
    epochs=FUSION_EPOCHS,
    thresholds=thresholds,
    selection_metric="micro_f1",
    experiment_name="SciBERT + TF-IDF",
    patience=2,
    early_stopping_metric="val_micro_f1",
):
    history = []
    best_val_score = -np.inf
    best_state_dict = None
    best_epoch = 0
    early_stop_counter = 0

    for epoch in range(epochs):
        print(f"\n======== {experiment_name} | Epoch {epoch + 1} / {epochs} ========")
        model.train()
        total_train_loss = 0.0
        train_probs = []
        train_targets = []

        progress = tqdm(train_dataloader, desc=f"Training {experiment_name}", leave=False)
        for batch in progress:
            batch = fusion_batch_to_device(batch, device)

            optimizer.zero_grad()
            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                tfidf_features=batch["tfidf_features"],
            )
            loss = criterion(logits, batch["labels"])
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            probs = torch.sigmoid(logits).detach().cpu().numpy()
            targets = batch["labels"].detach().cpu().numpy()
            train_probs.append(probs)
            train_targets.append(targets)
            progress.set_postfix(loss=f"{loss.item():.4f}")

        y_train_prob = np.vstack(train_probs)
        y_train_true = np.vstack(train_targets)
        train_loss = total_train_loss / len(train_dataloader)

        val_metrics, _, _, _, _ = evaluate_fusion_model(
            model,
            val_dataloader,
            criterion,
            thresholds=thresholds,
            selection_metric=selection_metric,
        )
        selected_threshold = val_metrics["threshold"]

        y_train_pred = (y_train_prob >= selected_threshold).astype(int)
        train_metrics = multilabel_metrics(y_train_true, y_train_pred)

        row = {
            "experiment": experiment_name,
            "epoch": epoch + 1,
            "threshold": selected_threshold,
            "train_loss": train_loss,
            "train_micro_f1": train_metrics["micro_f1"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_loss": val_metrics["loss"],
            "val_micro_f1": val_metrics["micro_f1"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_hamming_loss": val_metrics["hamming_loss"],
        }
        history.append(row)

        print(
            f"threshold={row['threshold']:.2f} "
            f"train_loss={row['train_loss']:.4f} "
            f"train_micro_f1={row['train_micro_f1']:.4f} "
            f"val_loss={row['val_loss']:.4f} "
            f"val_micro_f1={row['val_micro_f1']:.4f} "
            f"val_macro_f1={row['val_macro_f1']:.4f}"
        )

        current_val_score = row[early_stopping_metric]
        if current_val_score > best_val_score:
            best_val_score = current_val_score
            best_epoch = epoch + 1
            early_stop_counter = 0
            best_state_dict = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            print(f"Best model updated at epoch {best_epoch} ({early_stopping_metric}={best_val_score:.4f}).")
        else:
            early_stop_counter += 1
            print(f"No improvement. Patience: {early_stop_counter}/{patience}")

        if early_stop_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print(f"Reloaded best model from epoch {best_epoch} ({early_stopping_metric}={best_val_score:.4f}).")

    return pd.DataFrame(history)


In [22]:
fusion_training_stats = train_fusion_model(
    fusion_model,
    fusion_train_dataloader,
    fusion_val_dataloader,
    fusion_criterion,
    fusion_optimizer,
    fusion_scheduler,
    epochs=FUSION_EPOCHS,
    experiment_name="SciBERT + TF-IDF",
)
fusion_training_stats



======== SciBERT + TF-IDF | Epoch 1 / 10 ========


threshold=0.75 train_loss=0.6379 train_micro_f1=0.6147 val_loss=0.5264 val_micro_f1=0.6770 val_macro_f1=0.6563
Best model updated at epoch 1 (val_micro_f1=0.6770).

======== SciBERT + TF-IDF | Epoch 2 / 10 ========


threshold=0.80 train_loss=0.4630 train_micro_f1=0.7038 val_loss=0.5104 val_micro_f1=0.6875 val_macro_f1=0.6699
Best model updated at epoch 2 (val_micro_f1=0.6875).

======== SciBERT + TF-IDF | Epoch 3 / 10 ========


threshold=0.80 train_loss=0.3587 train_micro_f1=0.7657 val_loss=0.5116 val_micro_f1=0.7054 val_macro_f1=0.6859
Best model updated at epoch 3 (val_micro_f1=0.7054).

======== SciBERT + TF-IDF | Epoch 4 / 10 ========


threshold=0.75 train_loss=0.2695 train_micro_f1=0.8188 val_loss=0.5714 val_micro_f1=0.7136 val_macro_f1=0.6924
Best model updated at epoch 4 (val_micro_f1=0.7136).

======== SciBERT + TF-IDF | Epoch 5 / 10 ========


...

threshold=0.70 train_loss=0.1073 train_micro_f1=0.9267 val_loss=0.8545 val_micro_f1=0.7245 val_macro_f1=0.7026
No improvement. Patience: 1/2

======== SciBERT + TF-IDF | Epoch 8 / 10 ========


threshold=0.65 train_loss=0.0781 train_micro_f1=0.9474 val_loss=0.8997 val_micro_f1=0.7255 val_macro_f1=0.7024
Best model updated at epoch 8 (val_micro_f1=0.7255).

======== SciBERT + TF-IDF | Epoch 9 / 10 ========


threshold=0.45 train_loss=0.0580 train_micro_f1=0.9538 val_loss=0.9699 val_micro_f1=0.7253 val_macro_f1=0.7014
No improvement. Patience: 1/2

======== SciBERT + TF-IDF | Epoch 10 / 10 ========


threshold=0.60 train_loss=0.0436 train_micro_f1=0.9735 val_loss=1.0081 val_micro_f1=0.7261 val_macro_f1=0.7028
Best model updated at epoch 10 (val_micro_f1=0.7261).
Reloaded best model from epoch 10 (val_micro_f1=0.7261).


,experiment,epoch,threshold,train_loss,train_micro_f1,train_macro_f1,val_loss,val_micro_f1,val_macro_f1,val_hamming_loss
0,SciBERT + TF-IDF,1,0.75,0.637946,0.614665,0.593580,0.526427,0.676989,0.656331,0.070992
1,SciBERT + TF-IDF,2,0.80,0.462950,0.703783,0.687995,0.510403,0.687488,0.669926,0.066462
2,SciBERT + TF-IDF,3,0.80,0.358716,0.765698,0.756302,0.511607,0.705412,0.685871,0.061931
3,SciBERT + TF-IDF,4,0.75,0.269499,0.818774,0.813645,0.571388,0.713551,0.692362,0.062792
4,SciBERT + TF-IDF,5,0.75,0.199683,0.863213,0.863143,0.652078,0.716756,0.694608,0.060435
5,SciBERT + TF-IDF,6,0.65,0.147616,0.894558,0.895852,0.731741,0.725422,0.702738,0.059717
6,SciBERT + TF-IDF,7,0.70,0.107282,0.926708,0.929958,0.854541,0.724469,0.702627,0.058241
7,SciBERT + TF-IDF,8,0.65,0.078084,0.947443,0.949663,0.899674,0.725513,0.702359,0.058713
8,SciBERT + TF-IDF,9,0.45,0.057975,0.953777,0.954505,0.969939,0.725347,0.701387,0.060066
9,SciBERT + TF-IDF,10,0.60,0.043643,0.973515,0.974869,1.008101,0.726062,0.702848,0.057360


In [23]:
fusion_val_metrics, _, _, _, fusion_threshold_scores = evaluate_fusion_model(
    fusion_model,
    fusion_val_dataloader,
    fusion_criterion,
    thresholds=thresholds,
)
fusion_best_threshold = fusion_val_metrics["threshold"]

fusion_test_metrics, fusion_y_test_true, fusion_y_test_pred, fusion_y_test_prob, _ = evaluate_fusion_model(
    fusion_model,
    fusion_test_dataloader,
    fusion_criterion,
    threshold=fusion_best_threshold,
)

print(f"Best validation threshold: {fusion_best_threshold:.2f}")
fusion_test_metrics


Best validation threshold: 0.60


{'micro_f1': 0.7233685166269436,
 'macro_f1': 0.6951808134950513,
 'hamming_loss': 0.0582965003922469,
 'loss': 1.0685066027494954,
 'threshold': 0.6}

In [26]:
print(classification_report(
    fusion_y_test_true,
    fusion_y_test_pred,
    target_names=fusion_labels_18,
    zero_division=0,
))

#fusion_comparison = pd.concat([
#    comparison,
#    pd.DataFrame([{"experiment": "SciBERT + TF-IDF", **fusion_test_metrics}]),
#], ignore_index=True)
#fusion_comparison


              precision    recall  f1-score   support

        AGRI       0.70      0.71      0.71      1219
        BIOC       0.73      0.76      0.75      2480
        CENG       0.53      0.56      0.55       599
        CHEM       0.58      0.62      0.60       791
        COMP       0.64      0.61      0.62       903
        EART       0.80      0.82      0.81       838
        ENER       0.67      0.67      0.67       794
        ENGI       0.69      0.70      0.70      1789
        ENVI       0.71      0.75      0.73      1866
        IMMU       0.72      0.77      0.75       946
        MATE       0.77      0.78      0.78      1187
        MEDI       0.80      0.82      0.81      3004
        MULT       0.52      0.54      0.53       440
        NEUR       0.78      0.84      0.81       999
        PHAR       0.64      0.64      0.64       675
        PHYS       0.66      0.69      0.68      1059
        PSYC       0.63      0.62      0.63       574
        SOCI       0.78    

## Esperimento 4: SciBERT + TF-IDF solo body

Questo esperimento non usa l'abstract. SciBERT riceve i primi token di `body_clean` e il TF-IDF viene calcolato sugli stessi testi del body. Serve a verificare quanta informazione predittiva e contenuta nel corpo dell'articolo senza usare il riassunto.

In [27]:
BODY_ONLY_MODEL_NAME = "allenai/scibert_scivocab_uncased"
BODY_ONLY_MAX_LENGTH = MAX_LENGTH
BODY_ONLY_BATCH_SIZE = BATCH_SIZE
BODY_ONLY_TOKEN_LIMIT = 500
BODY_ONLY_TFIDF_MAX_FEATURES = 1000
BODY_ONLY_EPOCHS = 10

body_only_usecols = ["body_clean", "subjareas"]
body_only_df = pd.read_csv(DATA_PATH, usecols=body_only_usecols)
body_only_df = body_only_df.dropna(subset=body_only_usecols).copy()

def body_only_first_n_tokens(value, n=BODY_ONLY_TOKEN_LIMIT):
    text = safe_text(value)
    return " ".join(text.split()[:n])

body_only_df["body_text"] = body_only_df["body_clean"].apply(
    lambda value: body_only_first_n_tokens(value, BODY_ONLY_TOKEN_LIMIT)
)
body_only_df = body_only_df[body_only_df["body_text"].str.split().str.len() > 5].copy()

body_only_df["labels_list"] = body_only_df["subjareas"].apply(parse_labels)
body_only_df["labels_18_list"] = body_only_df["labels_list"].apply(remap_labels_18)

body_only_mlb_18 = MultiLabelBinarizer()
body_only_y_18 = body_only_mlb_18.fit_transform(body_only_df["labels_18_list"]).astype(np.float32)
body_only_labels_18 = body_only_mlb_18.classes_

print(f"Dataset body-only: {body_only_df.shape}")
print(f"Classi body-only: {len(body_only_labels_18)}")
print(body_only_labels_18)
assert list(body_only_labels_18) == list(labels_18), "Ordine label diverso rispetto agli esperimenti precedenti"


Dataset body-only: (38711, 5)
Classi body-only: 18
['AGRI' 'BIOC' 'CENG' 'CHEM' 'COMP' 'EART' 'ENER' 'ENGI' 'ENVI' 'IMMU'
 'MATE' 'MEDI' 'MULT' 'NEUR' 'PHAR' 'PHYS' 'PSYC' 'SOCI']


In [28]:
body_only_indices = np.arange(len(body_only_df))

body_only_idx_train_full, body_only_idx_test, body_only_y_train_full, body_only_y_test = train_test_split(
    body_only_indices,
    body_only_y_18,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
)

body_only_idx_train, body_only_idx_val, body_only_y_train, body_only_y_val = train_test_split(
    body_only_idx_train_full,
    body_only_y_train_full,
    test_size=0.10,
    random_state=SEED,
    shuffle=True,
)

body_only_X_train = body_only_df.iloc[body_only_idx_train]["body_text"].tolist()
body_only_X_val = body_only_df.iloc[body_only_idx_val]["body_text"].tolist()
body_only_X_test = body_only_df.iloc[body_only_idx_test]["body_text"].tolist()

body_only_tfidf_vectorizer = TfidfVectorizer(
    max_features=BODY_ONLY_TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
)

body_only_tfidf_train = torch.from_numpy(
    body_only_tfidf_vectorizer.fit_transform(body_only_X_train).astype(np.float32).toarray()
)
body_only_tfidf_val = torch.from_numpy(
    body_only_tfidf_vectorizer.transform(body_only_X_val).astype(np.float32).toarray()
)
body_only_tfidf_test = torch.from_numpy(
    body_only_tfidf_vectorizer.transform(body_only_X_test).astype(np.float32).toarray()
)

print(f"train: {len(body_only_X_train)}")
print(f"val:   {len(body_only_X_val)}")
print(f"test:  {len(body_only_X_test)}")
print(f"TF-IDF body-only dim: {body_only_tfidf_train.shape[1]}")


train: 24387
val:   2710
test:  11614
TF-IDF body-only dim: 1000


In [29]:
body_only_tokenizer = AutoTokenizer.from_pretrained(BODY_ONLY_MODEL_NAME, cache_dir=CACHE_DIR)

body_only_train_dataset = ArticleBertTfidfDataset(
    body_only_X_train,
    body_only_y_train,
    body_only_tfidf_train,
    body_only_tokenizer,
    BODY_ONLY_MAX_LENGTH,
)
body_only_val_dataset = ArticleBertTfidfDataset(
    body_only_X_val,
    body_only_y_val,
    body_only_tfidf_val,
    body_only_tokenizer,
    BODY_ONLY_MAX_LENGTH,
)
body_only_test_dataset = ArticleBertTfidfDataset(
    body_only_X_test,
    body_only_y_test,
    body_only_tfidf_test,
    body_only_tokenizer,
    BODY_ONLY_MAX_LENGTH,
)

body_only_train_dataloader = DataLoader(
    body_only_train_dataset,
    batch_size=BODY_ONLY_BATCH_SIZE,
    shuffle=True,
)
body_only_val_dataloader = DataLoader(
    body_only_val_dataset,
    batch_size=BODY_ONLY_BATCH_SIZE,
    shuffle=False,
)
body_only_test_dataloader = DataLoader(
    body_only_test_dataset,
    batch_size=BODY_ONLY_BATCH_SIZE,
    shuffle=False,
)


In [30]:
# Libera VRAM dagli esperimenti precedenti, se sono ancora caricati.
try:
    fusion_model.to("cpu")
except NameError:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()

body_only_model = SciBertTfidfClassifier(
    BODY_ONLY_MODEL_NAME,
    num_labels=len(body_only_labels_18),
    tfidf_dim=body_only_tfidf_train.shape[1],
).to(device)

body_only_pos_weight_18, body_only_class_weights_df = compute_pos_weight(body_only_y_train)
body_only_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.as_tensor(body_only_pos_weight_18, dtype=torch.float32, device=device)
)
body_only_optimizer = torch.optim.AdamW(body_only_model.parameters(), lr=2e-5, eps=1e-8)
body_only_total_steps = len(body_only_train_dataloader) * BODY_ONLY_EPOCHS
body_only_scheduler = get_linear_schedule_with_warmup(
    body_only_optimizer,
    num_warmup_steps=0,
    num_training_steps=body_only_total_steps,
)

body_only_model


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34722.79it/s]
[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SciBertTfidfClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31090, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elemen

In [31]:
body_only_training_stats = train_fusion_model(
    body_only_model,
    body_only_train_dataloader,
    body_only_val_dataloader,
    body_only_criterion,
    body_only_optimizer,
    body_only_scheduler,
    epochs=BODY_ONLY_EPOCHS,
    experiment_name="SciBERT + TF-IDF body only",
)
body_only_training_stats



======== SciBERT + TF-IDF body only | Epoch 1 / 10 ========


threshold=0.75 train_loss=0.6343 train_micro_f1=0.6167 val_loss=0.5316 val_micro_f1=0.6583 val_macro_f1=0.6398
Best model updated at epoch 1 (val_micro_f1=0.6583).

======== SciBERT + TF-IDF body only | Epoch 2 / 10 ========


threshold=0.80 train_loss=0.4736 train_micro_f1=0.6977 val_loss=0.4977 val_micro_f1=0.6923 val_macro_f1=0.6764
Best model updated at epoch 2 (val_micro_f1=0.6923).

======== SciBERT + TF-IDF body only | Epoch 3 / 10 ========


threshold=0.80 train_loss=0.3772 train_micro_f1=0.7534 val_loss=0.5128 val_micro_f1=0.6989 val_macro_f1=0.6815
Best model updated at epoch 3 (val_micro_f1=0.6989).

======== SciBERT + TF-IDF body only | Epoch 4 / 10 ========


threshold=0.75 train_loss=0.2911 train_micro_f1=0.8039 val_loss=0.5477 val_micro_f1=0.7057 val_macro_f1=0.6884
Best model updated at epoch 4 (val_micro_f1=0.7057).

======== SciBERT + TF-IDF body only | Epoch 5 / 10 ========


threshold=0.75 train_loss=0.2214 train_micro_f1=0.8478 val_loss=0.6566 val_micro_f1=0.7186 val_macro_f1=0.6987
Best model updated at epoch 5 (val_micro_f1=0.7186).

======== SciBERT + TF-IDF body only | Epoch 6 / 10 ========


threshold=0.70 train_loss=0.1652 train_micro_f1=0.8830 val_loss=0.7456 val_micro_f1=0.7211 val_macro_f1=0.7009
Best model updated at epoch 6 (val_micro_f1=0.7211).

======== SciBERT + TF-IDF body only | Epoch 7 / 10 ========


threshold=0.70 train_loss=0.1233 train_micro_f1=0.9134 val_loss=0.7997 val_micro_f1=0.7197 val_macro_f1=0.6994
No improvement. Patience: 1/2

======== SciBERT + TF-IDF body only | Epoch 8 / 10 ========


threshold=0.65 train_loss=0.0922 train_micro_f1=0.9354 val_loss=0.8801 val_micro_f1=0.7197 val_macro_f1=0.7020
No improvement. Patience: 2/2
Early stopping triggered at epoch 8.
Reloaded best model from epoch 6 (val_micro_f1=0.7211).


,experiment,epoch,threshold,train_loss,train_micro_f1,train_macro_f1,val_loss,val_micro_f1,val_macro_f1,val_hamming_loss
0,SciBERT + TF-IDF body only,1,0.75,0.634347,0.616698,0.595977,0.531603,0.658293,0.639757,0.078639
1,SciBERT + TF-IDF body only,2,0.80,0.473595,0.697750,0.682676,0.497650,0.692300,0.676399,0.066113
2,SciBERT + TF-IDF body only,3,0.80,0.377226,0.753403,0.744756,0.512761,0.698867,0.681509,0.064289
3,SciBERT + TF-IDF body only,4,0.75,0.291133,0.803942,0.798845,0.547739,0.705729,0.688448,0.064965
4,SciBERT + TF-IDF body only,5,0.75,0.221386,0.847762,0.847330,0.656550,0.718616,0.698709,0.060332
5,SciBERT + TF-IDF body only,6,0.70,0.165248,0.882993,0.885040,0.745554,0.721103,0.700890,0.060148
6,SciBERT + TF-IDF body only,7,0.70,0.123340,0.913407,0.917477,0.799666,0.719684,0.699422,0.059615
7,SciBERT + TF-IDF body only,8,0.65,0.092197,0.935418,0.938709,0.880119,0.719736,0.701955,0.059184


In [32]:
body_only_val_metrics, _, _, _, body_only_threshold_scores = evaluate_fusion_model(
    body_only_model,
    body_only_val_dataloader,
    body_only_criterion,
    thresholds=thresholds,
)
body_only_best_threshold = body_only_val_metrics["threshold"]

body_only_test_metrics, body_only_y_test_true, body_only_y_test_pred, body_only_y_test_prob, _ = evaluate_fusion_model(
    body_only_model,
    body_only_test_dataloader,
    body_only_criterion,
    threshold=body_only_best_threshold,
)

print(f"Best validation threshold: {body_only_best_threshold:.2f}")
body_only_test_metrics


Best validation threshold: 0.70


{'micro_f1': 0.7118688755108318,
 'macro_f1': 0.685671623212571,
 'hamming_loss': 0.06273080381914548,
 'loss': 0.7942314846415494,
 'threshold': 0.7}

In [33]:
print(classification_report(
    body_only_y_test_true,
    body_only_y_test_pred,
    target_names=body_only_labels_18,
    zero_division=0,
))

#body_only_base_comparison = fusion_comparison if "fusion_comparison" in globals() else comparison
#body_only_comparison = pd.concat([
#    body_only_base_comparison,
#    pd.DataFrame([{"experiment": "SciBERT + TF-IDF body only", **body_only_test_metrics}]),
#], ignore_index=True)
#body_only_comparison


              precision    recall  f1-score   support

        AGRI       0.65      0.70      0.67      1219
        BIOC       0.70      0.75      0.72      2480
        CENG       0.50      0.62      0.55       599
        CHEM       0.52      0.72      0.60       791
        COMP       0.62      0.64      0.63       903
        EART       0.81      0.80      0.80       838
        ENER       0.67      0.66      0.66       794
        ENGI       0.67      0.72      0.69      1789
        ENVI       0.68      0.78      0.73      1866
        IMMU       0.69      0.72      0.71       946
        MATE       0.74      0.83      0.78      1187
        MEDI       0.77      0.83      0.80      3004
        MULT       0.54      0.51      0.52       440
        NEUR       0.74      0.86      0.79       999
        PHAR       0.64      0.58      0.61       675
        PHYS       0.58      0.78      0.67      1059
        PSYC       0.64      0.61      0.63       574
        SOCI       0.75    